In [ ]:
import os
import glob
import time
import struct
import numpy as np
from pynq import Overlay, allocate, PL

print("=" * 70)
print(" STARTING UNIFIED LIDAR + CAMERA (BATCHED MM2S DMA) BENCHMARK")
print("=" * 70)

PL.reset()
ol = Overlay("depack_pipeline.bit") #bitstream

dma_lidar = ol.axi_dma_0
dma_camera = ol.axi_dma_1

print(f"[INFO] dma_lidar.sendchannel max transfer  : {dma_lidar.sendchannel._max_size:,} bytes")
print(f"[INFO] dma_lidar.recvchannel max transfer  : {dma_lidar.recvchannel._max_size:,} bytes")
print(f"[INFO] dma_camera.recvchannel max transfer : {dma_camera.recvchannel._max_size:,} bytes")

MAX_PAYLOAD_BYTES = 1400
NUM_SLOTS = 16
LIDAR_SLOT_SIZE = 1024 * MAX_PAYLOAD_BYTES
TOTAL_LIDAR_BYTES = NUM_SLOTS * LIDAR_SLOT_SIZE
CAMERA_SLOT_SIZE = 1024 * MAX_PAYLOAD_BYTES
TOTAL_CAMERA_BYTES = NUM_SLOTS * CAMERA_SLOT_SIZE

buf_l2_cam = allocate(shape=(TOTAL_CAMERA_BYTES,), dtype=np.uint8, cacheable=False)
buf_l2_lidar = allocate(shape=(TOTAL_LIDAR_BYTES,), dtype=np.uint8, cacheable=False)
buf_l2_cam[:] = 0
buf_l2_lidar[:] = 0

phys_cam = buf_l2_cam.physical_address
phys_lidar = buf_l2_lidar.physical_address

reg_l2 = ol.layer2_depack_0.register_map
ol.layer2_depack_0.write(reg_l2.ddr_mem_camera_1.address, phys_cam & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_camera_2.address, (phys_cam >> 32) & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_lidar_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_lidar_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer2_depack_0.write(0x00, 0x81)

reg_l1_cam = ol.layer1_camera_prep_0.register_map
ol.layer1_camera_prep_0.write(reg_l1_cam.ddr_mem_1.address, phys_cam & 0xFFFFFFFF)
ol.layer1_camera_prep_0.write(reg_l1_cam.ddr_mem_2.address, (phys_cam >> 32) & 0xFFFFFFFF)
ol.layer1_camera_prep_0.write(0x00,0x81)

reg_l1 = ol.layer1_depack_0.register_map
ol.layer1_depack_0.write(reg_l1.ddr_mem_meta_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_meta_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_points_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_points_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer1_depack_0.write(0x00, 0x81)

ol.packet_dispatcher_0.write(0x00,0x81);

CAMERA_DESC_WORDS = 9
MAX_WORDS_PER_LIDAR_FRAME = 262144
MAX_WORDS_PER_CAMERA_FRAME = (CAMERA_SLOT_SIZE // 4) + CAMERA_DESC_WORDS
PIXEL_FORMAT_NAMES = {0: "RGB", 1: "BGR", 2: "RGBA", 3: "GRAY"}

 STARTING UNIFIED LIDAR + CAMERA (BATCHED MM2S DMA) BENCHMARK
[INFO] dma_lidar.sendchannel max transfer  : 67,108,863 bytes
[INFO] dma_lidar.recvchannel max transfer  : 67,108,863 bytes
[INFO] dma_camera.recvchannel max transfer : 67,108,863 bytes


In [12]:
def dma_channel_error(channel):
    try:
        return bool(channel.error)
    except Exception:
        return None


def unpack_camera_field_desc(words9):
    raw = struct.pack("<9I", *[int(w) for w in words9])
    slot_index = raw[0]
    sensor_id = raw[1]
    frame_id = struct.unpack("<I", raw[2:6])[0]
    valid = raw[6]
    width = struct.unpack("<H", raw[7:9])[0]
    height = struct.unpack("<H", raw[9:11])[0]
    gop_size = raw[11]
    crf = raw[12]
    shared_field = raw[13]

    ts_bytes = raw[14:22]
    try:
        timestamp = struct.unpack(">d", ts_bytes)[0]
    except Exception:
        timestamp = 0.0

    bitstream_addr = struct.unpack("<Q", raw[22:30])[0]
    bitstream_length = struct.unpack("<I", raw[30:34])[0]
    compressed = bool(raw[34] & 0x01)

    profile_id = shared_field if compressed else None
    pixel_format = None if compressed else shared_field

    return {
        "slot_index": slot_index,
        "sensor_id": sensor_id,
        "frame_id": frame_id,
        "valid": valid,
        "compressed": compressed,
        "width": width,
        "height": height,
        "gop": gop_size,
        "crf": crf,
        "profile_id": profile_id,
        "pixel_format": pixel_format,
        "timestamp": timestamp,
        "bitstream_addr": bitstream_addr,
        "bitstream_length": bitstream_length,
    }


def load_golden_packed_words_dict(path):
    golden_dict = {}
    current_frame_id = None
    current_words = []

    if not os.path.exists(path):
        return golden_dict

    with open(path, "r") as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            if parts[0] == "FRAME":
                if current_frame_id is not None and current_words:
                    golden_dict[current_frame_id] = np.array(current_words, dtype=np.uint32)
                    current_words = []
                current_frame_id = int(parts[1])
            elif len(parts) == 1:
                current_words.append(int(parts[0], 16))

        if current_frame_id is not None and current_words:
            golden_dict[current_frame_id] = np.array(current_words, dtype=np.uint32)

    return golden_dict


def load_golden_camera_dict(golden_dir):
    cam_dict = {}
    for path in sorted(glob.glob(os.path.join(golden_dir, "golden_camera_*.h264"))):
        fname = os.path.basename(path)
        frame_id = int(fname.replace("golden_camera_", "").replace(".h264", ""))
        with open(path, "rb") as f:
            cam_dict[frame_id] = f.read()
    return cam_dict


def load_golden_camera_raw_dict(golden_dir):
    cam_dict = {}
    for path in sorted(glob.glob(os.path.join(golden_dir, "golden_camera_raw_*.bin"))):
        fname = os.path.basename(path)
        frame_id = int(fname.replace("golden_camera_raw_", "").replace(".bin", ""))
        with open(path, "rb") as f:
            cam_dict[frame_id] = f.read()
    return cam_dict

In [17]:
golden_dir = "golden_outputs"
interleaved_dat = os.path.join(golden_dir, "udp_input_interleaved.dat")
golden_lidar_dat = os.path.join(golden_dir, "golden_lidar.dat")

golden_lidar_dict = load_golden_packed_words_dict(golden_lidar_dat)
num_expected_lidar = len(golden_lidar_dict)
print(f"Loaded {num_expected_lidar} LiDAR golden reference frames: {list(golden_lidar_dict.keys())}")

golden_camera_dict = load_golden_camera_dict(golden_dir)
golden_camera_raw_dict = load_golden_camera_raw_dict(golden_dir)
num_expected_camera = len(golden_camera_dict) + len(golden_camera_raw_dict)
print(f"Loaded {len(golden_camera_dict)} compressed (H.264) camera golden frames: "
      f"{list(golden_camera_dict.keys())}")
print(f"Loaded {len(golden_camera_raw_dict)} raw camera golden frames: "
      f"{list(golden_camera_raw_dict.keys())}")

raw_packets, current_pkt = [], []
with open(interleaved_dat, "r") as f:
    for line in f:
        parts = line.split()
        if len(parts) != 3:
            continue
        word, keep, last = int(parts[0], 16), int(parts[1], 16), int(parts[2])
        current_pkt.append(word)
        if last:
            raw_packets.append(current_pkt)
            current_pkt = []
if current_pkt:
    raw_packets.append(current_pkt)

total_words = sum(len(pkt) for pkt in raw_packets)
print(f"Parsed {len(raw_packets):,} packets ({total_words:,} total DMA words)")

f_buf = allocate(shape=(total_words,), dtype=np.uint32, cacheable=False)
slices = []
offset = 0
for pkt in raw_packets:
    pkt_len = len(pkt)
    f_buf[offset: offset + pkt_len] = pkt
    slices.append((offset, offset + pkt_len))
    offset += pkt_len
f_buf.flush()

BATCH_PACKET_COUNT = 32
num_packets = len(slices)

batch_ranges = []
for i in range(0, num_packets, BATCH_PACKET_COUNT):
    start_word_idx = slices[i][0]
    end_packet_idx = min(i + BATCH_PACKET_COUNT - 1, num_packets - 1)
    end_word_idx = slices[end_packet_idx][1]
    batch_ranges.append((start_word_idx, end_word_idx, i, end_packet_idx))

print(f"[INFO] Created {len(batch_ranges)} batched DMA transfers "
      f"({BATCH_PACKET_COUNT} packets / ~{(BATCH_PACKET_COUNT * 1500) // 1024} KB per batch)")

rx_buf0 = allocate(shape=(MAX_WORDS_PER_LIDAR_FRAME,), dtype=np.uint32, cacheable=False)
rx_buf1 = allocate(shape=(MAX_WORDS_PER_LIDAR_FRAME,), dtype=np.uint32, cacheable=False)
rx_bufs = [rx_buf0, rx_buf1]

cam_rx_buf0 = allocate(shape=(MAX_WORDS_PER_CAMERA_FRAME,), dtype=np.uint32, cacheable=False)
cam_rx_buf1 = allocate(shape=(MAX_WORDS_PER_CAMERA_FRAME,), dtype=np.uint32, cacheable=False)
cam_rx_bufs = [cam_rx_buf0, cam_rx_buf1]

dma_lidar.sendchannel.start()
dma_lidar.recvchannel.start()
dma_camera.recvchannel.start()

captured_cam_frames = []
captured_lidar_frames = []
current_lidar_idx = 0
active_rx = 0
active_cam_rx = 0

if num_expected_lidar > 0:
    dma_lidar.recvchannel.transfer(rx_bufs[active_rx])
if num_expected_camera > 0:
    dma_camera.recvchannel.transfer(cam_rx_bufs[active_cam_rx])

Loaded 15 LiDAR golden reference frames: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014]
Loaded 15 compressed (H.264) camera golden frames: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]
Loaded 0 raw camera golden frames: []
Parsed 3,598 packets (1,292,902 total DMA words)
[INFO] Created 113 batched DMA transfers (32 packets / ~46 KB per batch)


In [18]:
def process_camera_completion():
    global active_cam_rx
    if not dma_camera.recvchannel.idle:
        return
    if len(captured_cam_frames) >= num_expected_camera:
        return

    curr_buf = cam_rx_bufs[active_cam_rx]
    if hasattr(curr_buf, "invalidate"):
        curr_buf.invalidate()

    fd = unpack_camera_field_desc(curr_buf[0:CAMERA_DESC_WORDS])

    bitstream_length = fd["bitstream_length"]
    bitstream_word_count = (bitstream_length + 3) // 4
    if CAMERA_DESC_WORDS + bitstream_word_count > len(curr_buf):
        print(f"   [WARN] Camera frame {fd['frame_id']} reports bitstream_length="
              f"{bitstream_length:,} B, which exceeds RX buffer. Skipping.")
        active_cam_rx = 1 - active_cam_rx
        if len(captured_cam_frames) < num_expected_camera:
            dma_camera.recvchannel.transfer(cam_rx_bufs[active_cam_rx])
        return

    bitstream_words = curr_buf[CAMERA_DESC_WORDS: CAMERA_DESC_WORDS + bitstream_word_count].copy()
    extracted_payload = bitstream_words.tobytes()[:bitstream_length]
    captured_cam_frames.append((fd, extracted_payload))

    active_cam_rx = 1 - active_cam_rx
    if len(captured_cam_frames) < num_expected_camera:
        dma_camera.recvchannel.transfer(cam_rx_bufs[active_cam_rx])


def process_lidar_completion():
    global current_lidar_idx, active_rx
    if dma_lidar.recvchannel.idle and current_lidar_idx < num_expected_lidar:
        curr_buf = rx_bufs[active_rx]
        if hasattr(curr_buf, "invalidate"):
            curr_buf.invalidate()

        raw_w0 = int(curr_buf[0])
        if raw_w0 == 0:
            return

        b0 = raw_w0 & 0xFF
        b1 = (raw_w0 >> 8) & 0xFF
        b2 = (raw_w0 >> 16) & 0xFF
        b3 = (raw_w0 >> 24) & 0xFF
        hw_frame_id = (b0 << 24) | (b1 << 16) | (b2 << 8) | b3

        if hw_frame_id not in golden_lidar_dict:
            alt_id = int(raw_w0 & 0xFFFF)
            if alt_id in golden_lidar_dict:
                hw_frame_id = alt_id

        golden_payload = golden_lidar_dict.get(hw_frame_id, None)

        if golden_payload is not None:
            expected_len = len(golden_payload)
            hw_payload = curr_buf[0: expected_len].copy()

            captured_lidar_frames.append((hw_frame_id, expected_len, hw_payload))
            curr_buf[0] = 0

            current_lidar_idx += 1
            active_rx = 1 - active_rx

            if current_lidar_idx < num_expected_lidar:
                dma_lidar.recvchannel.transfer(rx_bufs[active_rx])
        else:
            print(f"   [WARN] Unknown HW LiDAR Frame ID: {hw_frame_id} (Raw W0: 0x{raw_w0:08X})")


def service_hardware():
    process_camera_completion()
    process_lidar_completion()


def wait_send_and_service(sendchannel, timeout_s=2.0, label="MM2S sendchannel"):
    t0 = time.time()
    while not sendchannel.idle:
        service_hardware()
        err = dma_channel_error(sendchannel)
        if err:
            print(f"[ERROR] {label}: DMA reported an error state.")
            return False
        if time.time() - t0 > timeout_s:
            print(f"[TIMEOUT] {label} did not go idle within {timeout_s}s.")
            return False
    return True

In [19]:
t_start = time.time()
aborted = False

for batch_idx, (start, end, p_start, p_end) in enumerate(batch_ranges):
    dma_lidar.sendchannel.transfer(f_buf[start:end])

    if not wait_send_and_service(dma_lidar.sendchannel, timeout_s=2.0, label=f"MM2S Batch {batch_idx}"):
        print(f"[ABORT] Send stalled at batch {batch_idx} (Packets {p_start}..{p_end}).")
        aborted = True
        break

    print(f"  ... sent batch {batch_idx + 1}/{len(batch_ranges)} "
          f"(packets {p_end + 1}/{num_packets} | "
          f"LiDAR {current_lidar_idx}/{num_expected_lidar}, "
          f"Camera {len(captured_cam_frames)}/{num_expected_camera})")

if not aborted:
    t_drain = time.time()
    while ((current_lidar_idx < num_expected_lidar or len(captured_cam_frames) < num_expected_camera)
           and (time.time() - t_drain < 5.0)):
        service_hardware()

total_elapsed_sec = time.time() - t_start

  ... sent batch 1/113 (packets 32/3598 | LiDAR 0/15, Camera 0/15)
  ... sent batch 2/113 (packets 64/3598 | LiDAR 0/15, Camera 0/15)
  ... sent batch 3/113 (packets 96/3598 | LiDAR 0/15, Camera 0/15)
  ... sent batch 4/113 (packets 128/3598 | LiDAR 0/15, Camera 0/15)
  ... sent batch 5/113 (packets 160/3598 | LiDAR 0/15, Camera 0/15)
  ... sent batch 6/113 (packets 192/3598 | LiDAR 0/15, Camera 0/15)
  ... sent batch 7/113 (packets 224/3598 | LiDAR 0/15, Camera 1/15)
  ... sent batch 8/113 (packets 256/3598 | LiDAR 0/15, Camera 1/15)
  ... sent batch 9/113 (packets 288/3598 | LiDAR 0/15, Camera 1/15)
  ... sent batch 10/113 (packets 320/3598 | LiDAR 0/15, Camera 1/15)
  ... sent batch 11/113 (packets 352/3598 | LiDAR 0/15, Camera 1/15)
  ... sent batch 12/113 (packets 384/3598 | LiDAR 0/15, Camera 1/15)
  ... sent batch 13/113 (packets 416/3598 | LiDAR 0/15, Camera 1/15)
  ... sent batch 14/113 (packets 448/3598 | LiDAR 1/15, Camera 2/15)
  ... sent batch 15/113 (packets 480/3598 | Li

In [20]:
print("\n[VALIDATE] Comparing Outputs Against Golden References...")
print("=" * 70)

total_cam_passed = 0
total_cam_compressed_passed = 0
total_cam_raw_passed = 0
total_lidar_passed = 0

for fd, extracted_payload in captured_cam_frames:
    if fd["compressed"]:
        golden_bytes = golden_camera_dict.get(fd["frame_id"])
    else:
        golden_bytes = golden_camera_raw_dict.get(fd["frame_id"])

    match = False
    if golden_bytes is not None:
        match = (extracted_payload == golden_bytes)

    if match:
        total_cam_passed += 1
        if fd["compressed"]:
            total_cam_compressed_passed += 1
        else:
            total_cam_raw_passed += 1

    mode_str = "COMPRESSED (H.264)" if fd["compressed"] else "RAW"
    status = "[PASSED] 100% Match" if match else "[MISMATCH] detected"
    print(f"  [CAMERA OUT] Frame ID {fd['frame_id']} | Mode={mode_str} : {status}")

for frame_id, expected_words, hw_payload in captured_lidar_frames:
    golden_payload = golden_lidar_dict[frame_id]

    if len(hw_payload) == len(golden_payload) and np.array_equal(hw_payload, golden_payload):
        print(f"  [LIDAR OUT]  Frame ID {frame_id} ({expected_words:,} words) : [PASSED] 100% Match")
        total_lidar_passed += 1
    else:
        mismatches = np.where(hw_payload != golden_payload[:len(hw_payload)])[0]
        print(f"  [LIDAR OUT]  Frame ID {frame_id}: [MISMATCH] ({len(mismatches)}/{expected_words} words failed)")

print("\n" + "=" * 70)
print(" FINAL BENCHMARK SUMMARY")
print("=" * 70)
print(f" Aborted Early         : {aborted}")
print(f" Total Execution Time  : {total_elapsed_sec:.3f} s")
print(f" Camera Frame Results  : {total_cam_passed}/{len(captured_cam_frames)} Frames Passed "
      f"(Compressed: {total_cam_compressed_passed}/{len(golden_camera_dict)}, "
      f"Raw: {total_cam_raw_passed}/{len(golden_camera_raw_dict)})")
print(f" LiDAR Frame Results   : {total_lidar_passed}/{num_expected_lidar} Frames Passed (100% Word Match)")

if (not aborted and num_expected_lidar > 0 and total_lidar_passed == num_expected_lidar
        and num_expected_camera > 0 and total_cam_passed == num_expected_camera):
    print("\n OVERALL STATUS     : SUCCESS - 100% MATCH ACROSS ALL SENSORS!")
else:
    print("\n OVERALL STATUS     : PARTIAL MATCH / FAILED")
print("=" * 70)


[VALIDATE] Comparing Outputs Against Golden References...
  [CAMERA OUT] Frame ID 2000 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2001 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2002 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2003 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2004 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2005 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2006 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2007 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2008 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2009 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2010 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Frame ID 2011 | Mode=COMPRESSED (H.264) : [PASSED] 100% Match
  [CAMERA OUT] Fr